In [13]:
!pwd


/Users/mo/Desktop/repos/NYC_Taxi_analysis/src/nyc_taxi_analysis/jobs


In [24]:
import glob

# Find all matching files on your drive using Python
file_list = glob.glob(f"{RAW}/yellow_tripdata_*.csv")

# Print to verify Python found them
print(f"Files found by python: {file_list}")


Files found by python: ['../../../data/raw/yellow_tripdata_2019-11.csv', '../../../data/raw/yellow_tripdata_2019-05.csv', '../../../data/raw/yellow_tripdata_2019-04.csv', '../../../data/raw/yellow_tripdata_2019-10.csv', '../../../data/raw/yellow_tripdata_2019-06.csv', '../../../data/raw/yellow_tripdata_2019-12.csv', '../../../data/raw/yellow_tripdata_2019-07.csv', '../../../data/raw/yellow_tripdata_2019-03.csv', '../../../data/raw/yellow_tripdata_2019-02.csv', '../../../data/raw/yellow_tripdata_2019-01.csv', '../../../data/raw/yellow_tripdata_2020-03.csv', '../../../data/raw/yellow_tripdata_2020-02.csv', '../../../data/raw/yellow_tripdata_2020-01.csv', '../../../data/raw/yellow_tripdata_2020-05.csv', '../../../data/raw/yellow_tripdata_2020-04.csv', '../../../data/raw/yellow_tripdata_2020-06.csv', '../../../data/raw/yellow_tripdata_2019-09.csv', '../../../data/raw/yellow_tripdata_2019-08.csv']


In [19]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("ingest-bronze").getOrCreate()

RAW = "../../../data/raw"

In [20]:
## schema enforcement
trip_schema = T.StructType([
    T.StructField("VendorID", T.IntegerType()),
    T.StructField("tpep_pickup_datetime", T.TimestampType()),
    T.StructField("tpep_dropoff_datetime", T.TimestampType()),
    T.StructField("passenger_count", T.IntegerType()),
    T.StructField("trip_distance", T.DoubleType()),
    T.StructField("RatecodeID", T.IntegerType()),
    T.StructField("store_and_fwd_flag", T.StringType()),
    T.StructField("PULocationID", T.IntegerType()),
    T.StructField("DOLocationID", T.IntegerType()),
    T.StructField("payment_type", T.IntegerType()),
    T.StructField("fare_amount", T.DoubleType()),
    T.StructField("extra", T.DoubleType()),
    T.StructField("mta_tax", T.DoubleType()),
    T.StructField("tip_amount", T.DoubleType()),
    T.StructField("tolls_amount", T.DoubleType()),
    T.StructField("improvement_surcharge", T.DoubleType()),
    T.StructField("total_amount", T.DoubleType()),
    T.StructField("congestion_surcharge", T.DoubleType()),
    T.StructField("_corrupt", T.StringType()),
])

In [ ]:
raw = (spark.read
       .option("header", True)
       .option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt")
       .schema(trip_schema)
       .csv(f"{RAW}/yellow_tripdata_*.csv"))

26/10/24 12:23:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../../../data/raw/yellow_tripdata_*.csv.
java.io.FileNotFoundException: File ../../../data/raw/yellow_tripdata_*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

In [ ]:
raw.show(5)
file_month = r"(\d{4})-(\d{2})\.csv$"

trips = (raw
         .withColumn("source_file", F.col("_metadata.file_path"))
         .withColumn("year",  F.regexp_extract("source_file", file_month, 1).cast("int"))
         .withColumn("month", F.regexp_extract("source_file", file_month, 2).cast("int"))
         .withColumn("ingested_at", F.current_timestamp()))

In [ ]:
trips.show(5)

In [ ]:
# Calling trips.cache() tells 
# Spark to save this dataset in 
# your computer's memory (RAM) 
# instead of reading it from your hard drive over and over again.
trips.cache()

In [ ]:
good = trips.filter(F.col("_corrupt").isNull()).drop("_corrupt")
bad  = trips.filter(F.col("_corrupt").isNotNull()).select("_corrupt", "source_file", "year", "month", "ingested_at")

In [ ]:
bad.show(5)